### `Imports`

In [1]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

I0000 00:00:1789237563.220648  372070 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789237563.221000  372070 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789237563.252390  372070 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789237564.091354  372070 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
test_df = pd.read_csv("../data/raw/test.csv")
sample_submission = pd.read_csv("../data/raw/sample_submission.csv")

print("Test shape:", test_df.shape)
print("Submission template:", sample_submission.shape)

Test shape: (4277, 13)
Submission template: (4277, 2)


In [3]:
passenger_ids = test_df["PassengerId"].copy()

### `Same feature engineering function from final training`

In [4]:
def feature_engineer(df):
    df = df.copy() # We do not want to modify the actual dataFrame

    # Extract group ID
    df["GroupID"] = df["PassengerId"].str.split("_").str[0]

    # Extract group size
    group_count = df["GroupID"].value_counts()
    df["GroupSize"] = df["GroupID"].map(group_count)

    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

    # Create a total spending column
    df["TotalSpending"] = df[spending_cols].sum(axis=1)

    # Extract Cabin into three new useful columns
    df[['Deck', 'CabinNumber', 'Side']] = df['Cabin'].str.split('/', expand=True)

    # Convert CabinNumber to numeric, coercing errors to NaN
    df["CabinNumber"] = pd.to_numeric(
                            df["CabinNumber"],
                            errors="coerce")


    # Drop unecessaary columns 
    df = df.drop(columns=["GroupID","PassengerId","Cabin","Name"])
    

    return df

In [5]:
X_test = feature_engineer(test_df)

print("Test shape after feature engineering:", X_test.shape)
X_test.head()

Test shape after feature engineering: (4277, 15)


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,GroupSize,TotalSpending,Deck,CabinNumber,Side
0,Earth,True,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,1,0.0,G,3.0,S
1,Earth,False,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,1,2832.0,F,4.0,S
2,Europa,True,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,1,0.0,C,0.0,S
3,Europa,False,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,1,7418.0,C,1.0,S
4,Earth,False,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,1,645.0,F,5.0,S


### `Load the saved preprocessor`

In [6]:
preprocessor = joblib.load(
    "../models/preprocessor.joblib"
)

In [7]:
X_test_processed = preprocessor.transform(X_test)

print("Processed test shape:", X_test_processed.shape)
print("NaNs:", np.isnan(X_test_processed).sum())

Processed test shape: (4277, 29)
NaNs: 0


### `Load final model`

In [8]:
final_model = tf.keras.models.load_model(
    "../models/final_spaceship_titanic.keras"
)

E0000 00:00:1789237565.184247  372070 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789237565.184557  372597 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789237565.201404  372070 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [9]:
final_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 16)             │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,877 (7.34 KB)

 Trainable params: 625 (2.44 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,252 (4.89 KB)

### `Predict Kaggle passengers`

In [10]:
probabilities = final_model.predict(
    X_test_processed
)

print(probabilities.shape)
print(probabilities[:10])

134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 530us/step
(4277, 1)
[[0.53635955]
 [0.01781183]
 [0.99847734]
 [0.992533  ]
 [0.5990579 ]
 [0.6444788 ]
 [0.9878021 ]
 [0.99452966]
 [0.99723434]
 [0.44562334]]


In [11]:
predictions = probabilities.ravel() >= 0.5
print(predictions[:10])

[ True False  True  True  True  True  True  True  True False]


### `Create the submission`

In [17]:
submission = sample_submission.copy()

submission["PassengerId"] = passenger_ids
submission["Transported"] = predictions

submission.head(10)

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
5,0027_01,True
6,0029_01,True
7,0032_01,True
8,0032_02,True
9,0033_01,False


In [18]:
print(submission.shape)

print(submission["Transported"].value_counts())

print(submission.isna().sum())

(4277, 2)
Transported
True     2362
False    1915
Name: count, dtype: int64
PassengerId    0
Transported    0
dtype: int64


In [24]:
assert submission.shape[0] == test_df.shape[0], f"Submission shape does not match test shape"
assert submission["PassengerId"].is_unique, f"PassengerId is not unique"
assert submission["PassengerId"].equals(test_df["PassengerId"]), f"PassengerId does not match test PassengerId"
assert submission["Transported"].isna().sum() == 0
print("Submission checks passed.")

Submission checks passed.


In [25]:
# Save the submission file
submission.to_csv(
    "../artifacts/submission.csv",
    index=False
)

print("submission.csv created successfully.")

submission.csv created successfully.


### `Final verification`

In [28]:
check_submission = pd.read_csv("../artifacts/submission.csv")
check_submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [29]:
print(check_submission.shape)
print(check_submission.dtypes)

(4277, 2)
PassengerId     str
Transported    bool
dtype: object
